# Plan Hang Analysis

Diagnose a **hung or very slow** `terraform plan` from **`TF_LOG=json`** trace output.

Looks for:

- **Graph wait loops** — `dag/walk: vertex ... is waiting for ...` (spinning in the graph)
- **Stuck refresh** — refresh started but not completing (with **open duration** to last log line)
- **Vertex churn** — graph vertices revisited repeatedly
- **SDK pressure** — when API traffic spiked (timeline), plus retry/404/429 if elevated
- **Refresh timing** — completed refresh durations plus open/stuck refreshes

```bash
export TF_LOG=json
export TF_LOG_PATH=plan-hang.log
# optional: GENESYSCLOUD_SDK_DEBUG=true GENESYSCLOUD_SDK_DEBUG_FORMAT=Json
terraform plan
export TERRAFORM_LOG_PATH=plan-hang.log
```

For completed plans use `plan/performance-analysis.ipynb`. Run `whatisit.ipynb` first if unsure.


In [ ]:
import sys
from pathlib import Path

for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / "commonlib" / "config.py").is_file():
        _notebooks_root = _root
        break
    if (_root / "notebooks" / "commonlib" / "config.py").is_file():
        _notebooks_root = _root / "notebooks"
        break
else:
    raise RuntimeError(
        "Could not find notebooks/commonlib/. Start Jupyter from notebooks/ "
        "or open a notebook under export/, plan/, apply/, sdk-plan/, or the notebooks root."
    )

if str(_notebooks_root) not in sys.path:
    sys.path.insert(0, str(_notebooks_root))

from commonlib.notebook_setup import setup

setup()

import matplotlib.pyplot as plt
import pandas as pd

import commonlib.config as cfg
import commonlib.gencharts as gencharts
import commonlib.prep_hang_data as hang
import commonlib.prep_plan_log_data as prep_plan_log_data


In [ ]:
TAIL_MINUTES = 5
MIN_REPEAT = 3
WORKFLOW = "plan"

c = cfg.Config()
print(c.TERRAFORM_LOG_PATH)

classification, records, counters = hang.load_hang_scan(c.TERRAFORM_LOG_PATH, tail_minutes=TAIL_MINUTES)
if not records:
    raise ValueError("No JSON log lines found. Capture with TF_LOG=json and run whatisit.ipynb first.")

normalized_records = prep_plan_log_data.normalize_records(records)
df_refresh, df_merged_refresh, df_open_refresh = prep_plan_log_data.refresh_timing_dataframes(
    normalized_records
)

if not classification.is_plan and not counters.dag_waits and not counters.refresh_starts:
    raise ValueError(
        "No plan trace activity found. Use plan/hang-analysis.ipynb on a TF_LOG plan capture, "
        "or sdk-plan/plan-analysis.ipynb for terraform plan -json UI output."
    )

summary = hang.hang_summary_for_workflow(
    counters,
    tail_minutes=TAIL_MINUTES,
    workflow=WORKFLOW,
    min_count=MIN_REPEAT,
    classification=classification,
)


## Summary

In [ ]:
print(f"Parsed lines: {summary['parsed_lines']:,}")
if summary['duration_minutes'] is not None:
    print(f"Log span: {summary['duration_minutes']:.1f} minutes")
if summary['first_timestamp']:
    print(f"Time range: {summary['first_timestamp']} → {summary['last_timestamp']}")
print(f"Tail window: last {summary['tail_minutes']:.0f} minutes")
print()
print(f"Primary suspect: {summary['primary_summary']}")
if summary['primary_detail']:
    print(f"  {summary['primary_detail']}")
print()
print(
    f"Plan: {summary['dag_wait_pairs']:,} dag wait pairs, "
    f"{summary['sdk_retry_endpoints']:,} retry endpoints, "
        f"{summary['sdk_404_endpoints']:,} 404 endpoints, "
    f"{summary['sdk_429_response_count']:,} 429 responses, "
    f"{summary['sdk_429_wait_minutes']:.1f} min 429 wait"
)
if summary['sdk_status_codes']:
    print(f"SDK status codes: {summary['sdk_status_codes']}")
print()
hang.display_issue_attribution(hang.build_issue_attribution(counters, summary, WORKFLOW))


## Refresh timing

How long resource refreshes took before the hang, and **open refreshes** (started but not completed) measured to the last log line.

In [ ]:
print(
    f"Completed refresh pairs: {len(df_merged_refresh):,}  |  "
    f"Open refreshes (no complete): {len(df_open_refresh):,}"
)

if df_open_refresh.empty:
    print("No open refreshes — every refresh_start had a matching refresh_complete.")
else:
    display(
        df_open_refresh[
            [
                "resource",
                "resource_type",
                "start_timestamp",
                "open_minutes",
                "run",
            ]
        ].head(20)
    )

In [ ]:
if df_merged_refresh.empty:
    print("No matched refresh_start/refresh_complete pairs to chart.")
else:
    gencharts.generate_duration_by_resource_type(df_merged_refresh, metric="total", top_n=10)
    gencharts.generate_duration_by_resource_type(df_merged_refresh, metric="average", top_n=10)

In [ ]:
if df_merged_refresh.empty:
    print("No matched refresh_start/refresh_complete pairs to list.")
else:
    display(
        df_merged_refresh[
            [
                "resource",
                "resource_type",
                "start_timestamp",
                "end_timestamp",
                "time_diff_minutes",
                "run",
            ]
        ]
        .sort_values(by="time_diff_minutes", ascending=False)
        .head(20)
    )

## Ranked hang suspects

In [ ]:
pd.DataFrame([
    {"category": v.category, "summary": v.summary, "detail": v.detail, "score": v.score}
    for v in summary["verdicts"]
]) if summary["verdicts"] else "No strong hang patterns detected (try lowering MIN_REPEAT)."


## Graph wait loops (spinning)

Resources Terraform keeps waiting on.

In [ ]:
df_blocked = hang.blocked_resources_dataframe(counters, min_count=MIN_REPEAT)
df_blocked.head(20) if not df_blocked.empty else "No repeated dag/walk wait lines found."

In [ ]:
if not df_blocked.empty:
    plot_df = df_blocked.head(15).sort_values("wait_messages")
    plot_df = plot_df.assign(
        label=plot_df["waiting_for"].str.replace(r"^module\.", "", regex=True).str.slice(0, 60)
    )
    plt.figure(figsize=(12, 6))
    plt.barh(plot_df["label"], plot_df["wait_messages"], color="tab:orange")
    plt.xlabel("dag/walk wait lines")
    plt.title("Top resources the graph is waiting on")
    plt.tight_layout()

## Stuck refresh and vertex churn

In [ ]:
hang.refresh_imbalance_dataframe(counters, min_gap=2).head(20)


In [ ]:
hang.vertex_churn_dataframe(counters, min_count=10).head(20)


## SDK pressure

**When** the provider hammered the API (timeline), plus compact whole-log rates. Retry, 404, and 429 detail appears only when counts are elevated. Full tables are in `*-report.json` — see **`HOW-TO-READ-RESULTS.md`**.


In [ ]:
sdk_pressure = hang.display_sdk_pressure(counters, summary, min_retry=MIN_REPEAT, min_404=2)
df_sdk_rates = sdk_pressure["df_sdk_rates"]
df_sdk_rates_by_type = sdk_pressure["df_sdk_rates_by_type"]
df_retries = sdk_pressure["df_retries"]
df_404 = sdk_pressure["df_404"]


## Tail activity (last few minutes)

What the log was doing right before capture.

In [ ]:
hang.tail_messages_dataframe(counters, top_n=25)


## Export report

Writes `{capture-stem}-report.json` next to the log. See **`HOW-TO-READ-RESULTS.md`** at the repo root.


In [ ]:
import commonlib.run_report as run_report

df_blocked = hang.blocked_resources_dataframe(counters, min_count=MIN_REPEAT)
df_retries = hang.sdk_retry_dataframe(counters, min_count=MIN_REPEAT)
df_404 = hang.sdk_not_found_dataframe(counters, min_count=2)

df_sdk_rates = hang.sdk_call_rates_dataframe(
    counters, duration_minutes=summary.get("duration_minutes")
)
df_sdk_rates_by_type = hang.sdk_call_rates_by_resource_type_dataframe(
    counters, duration_minutes=summary.get("duration_minutes")
)

run_report.write_hang_report(
    c.TERRAFORM_LOG_PATH,
    WORKFLOW,
    {
        **run_report.issue_attribution_bundle(counters, summary, WORKFLOW),
        "open_refreshes": run_report.dataframe_records(df_open_refresh),
        "completed_refreshes": run_report.dataframe_records(df_merged_refresh),
        "ranked_verdicts": run_report.verdict_records(summary["verdicts"]),
        "graph_wait_loops": run_report.dataframe_records(df_blocked),
        "refresh_imbalance": run_report.dataframe_records(
            hang.refresh_imbalance_dataframe(counters, min_gap=2)
        ),
        "vertex_churn": run_report.dataframe_records(
            hang.vertex_churn_dataframe(counters, min_count=10)
        ),
        **run_report.sdk_report_sections(
            counters, duration_minutes=summary.get("duration_minutes")
        ),
        "tail_messages": run_report.dataframe_records(
            hang.tail_messages_dataframe(counters, top_n=25), head=25
        ),
    },
)